# Relatório Técnico — Tech Challenge Fase 4

**Tema:** Monitoramento multimodal de pacientes em reabilitação/UTI com detecção de anomalias em tempo real

**Curso:** Pós FIAP — 8IADT · Fase 4  
**Projeto:** TechChallenge Multimodais  
**Stack:** local e gratuita (MediaPipe, YOLOv8, Whisper, PyOD, Ollama + LoRA médico)

> **Aviso educacional:** este sistema e o modelo LLM médico associado são **educacionais**.  
> Não substituem avaliação, diagnóstico ou tratamento por profissional de saúde.

## 1. Problema e objetivo

Com a IA já integrada a processos médicos (exames, documentos e apoio à decisão), o hospital deseja **monitorar continuamente** os pacientes por meio de dados multimodais — **vídeo, áudio e sinais vitais/texto clínico** — para identificar sinais precoces de risco.

Neste trabalho, o fio condutor é um **paciente fictício** em reabilitação / UTI simulada. Isso encaixa os três blocos do enunciado:

| Modalidade | Cenário | O que detectamos |
|---|---|---|
| **Vídeo** | Sessões de fisioterapia | Padrões de movimento / postura anômalos |
| **Áudio** | Consultas / check-ins de voz | Fadiga, disartria, termos críticos |
| **Anomalias** | Sinais vitais + prescrição | Desvios em HR/SpO₂/PA e evolução inesperada do tratamento |

### Objetivos

1. Analisar e **fundir** diferentes tipos de dados médicos (vídeo, áudio, séries temporais e texto de prescrição).
2. Aplicar **detecção de anomalias** (incluindo simulação de tempo quase real por janelas).
3. Gerar **alertas e resumos clínicos** para a equipe médica via motor de LLM local.
4. Documentar a solução completa (código, relatório e demo em vídeo).

## 2. Arquitetura multimodal

Cada modalidade tem processamento especializado; os scores convergem na fusão e o motor de alertas (LLM) produz o resumo para a equipe.

```text
┌─────────────────┐   ┌─────────────────┐   ┌─────────────────┐
│ Vídeo           │   │ Áudio           │   │ Sinais vitais   │
│ (fisioterapia)  │   │ (consulta)      │   │ HR / SpO₂ / PA  │
└────────┬────────┘   └────────┬────────┘   └────────┬────────┘
         │                     │                     │
         ▼                     ▼                     ▼
┌─────────────────┐   ┌─────────────────┐   ┌─────────────────┐
│ Análise postural│   │ Transcrição NLP │   │ Anomalias       │
│ MediaPipe+YOLOv8│   │ Whisper+Transform│  │ IsolationForest │
└────────┬────────┘   └────────┬────────┘   └────────┬────────┘
         │                     │                     │
         └──────────┬──────────┴──────────┬──────────┘
                    ▼                     ▼
           ┌────────────────┐    ┌────────────────┐
           │ Fusão multimodal│───▶│ Motor de alertas│
           │ (3 scores)      │    │ LLM local      │
           └────────────────┘    └────────┬───────┘
                                          ▼
                                 ┌────────────────┐
                                 │ Equipe médica  │
                                 └────────────────┘
```

### Mapeamento no repositório

| Etapa | Pacote |
|---|---|
| Pose / anomalia motora | `src/video/` (`pose_estimation.py`, `anomaly_report.py`) |
| Transcrição / fala | `src/audio/` (`transcription.py`, `speech_analysis.py`) |
| Vitais / prescrição | `src/vitals/` (`anomaly_detection.py`, `prescription_check.py`) |
| Fusão de risco | `src/fusion/risk_fusion.py` |
| Relatório LLM | `src/llm/ollama_report.py` |
| Notificações | `src/alerts/notifier.py` |
| Modelos locais | `.env` → `LOCAL_LLAMA_MODEL_PATH`, `MEDICAL_ADAPTER_PATH` |

## 3. Requisitos do enunciado × solução adotada

O PDF da Fase 4 cita **Azure Cognitive Services**. Optamos por **stack gratuita local**, com equivalência funcional documentada abaixo (aceitável academicamente desde que justificado).

| Requisito (PDF) | Solução local | Status |
|---|---|---|
| Análise de vídeo (fisio/cirurgia) | MediaPipe Pose + YOLOv8 | Planejado (`src/video`) |
| OpenPose / postura | **MediaPipe Pose** (alternativa ao OpenPose) | Planejado |
| YOLOv8 (objetos / áreas) | Ultralytics YOLOv8 | Planejado |
| Relatório de desvios no procedimento | `anomaly_report.py` + LLM | Planejado |
| Áudio de consultas | Whisper (STT) | Planejado (`src/audio`) |
| Azure Speech to Text | **Whisper** | Equivalente local |
| Azure Text Analytics (termos/sentimento) | **Transformers** + regras de termos críticos | Equivalente local |
| Fadiga / disartria | Features acústicas (`speech_analysis.py`) | Planejado |
| Anomalias em vitais | Isolation Forest / PyOD | Planejado (`src/vitals`) |
| Evolução de prescrições | `prescription_check.py` (texto/regras) | Planejado |
| Alertas à equipe | Fusão + Ollama + `notifier.py` | Planejado |
| Serviços em nuvem Azure | Stack local + Ollama; LoRA médico no HF | Substituído / justificado |

### Tabela de equivalência Azure → local

| Azure (enunciado) | Equivalente gratuito |
|---|---|
| Speech to Text | OpenAI Whisper |
| Text Analytics | Hugging Face Transformers + léxico clínico |
| Cognitive / resumo | Ollama (`llama3.2`) + adapter LoRA médico |
| Pipeline gerenciado | Scripts Python + notebooks neste repositório |

## 4. Datasets e justificativa acadêmica

### Estratégia do MVP

Para validar o pipeline com rapidez e facilitar o relatório técnico, o MVP usa **dados controlados**:

| Modalidade | MVP (agora) | Próximo passo (público) |
|---|---|---|
| Vitais | Séries **sintéticas** com anomalias injetadas | PhysioNet (MIT-BIH, MIMIC Waveform) |
| Áudio | Amostra curta própria ou clip público pequeno | Coswara / Saarbrücken / Parkinson (PhysioNet, UCI) |
| Vídeo | Webcam própria (exercício simulado) | UCF101 / NTU RGB+D adaptados; documentar simulação |

### Por que isso é aceitável

- Datasets clínicos públicos de **fisioterapia completa** são raros.
- Séries sintéticas com anomalias **conhecidas** permitem medir se o detector acerta (ground truth controlado).
- Vídeo próprio / ação humana genérica adaptada é prática comum em trabalhos acadêmicos — desde que as **limitações** estejam explícitas no relatório.

### Fontes sugeridas pelo enunciado / literatura

- PhysioNet: https://physionet.org/
- AudioSet (auxiliar): https://research.google.com/audioset/
- Adapter médico: https://huggingface.co/StefanieFranco/llama3-medical-fine-tuning

Dados brutos → `data/raw/` · processados → `data/processed/`.

## 5. Stack e modelos

| Camada | Tecnologia | Papel |
|---|---|---|
| Vídeo | MediaPipe Pose, YOLOv8, OpenCV | Landmarks articulares e eventos visuais |
| Áudio | Whisper, librosa | Transcrição e features de fala |
| NLP | Transformers | Termos críticos / sentimento (proxy Text Analytics) |
| Vitais | scikit-learn Isolation Forest, PyOD | Anomalias em séries temporais |
| Fusão | Score ponderado (`risk_fusion.py`) | Combina risco vídeo + áudio + vitais |
| LLM demo | Ollama `llama3.2` | Resumo/alerta rápido |
| LLM médico | Meta-Llama-3-8B-Instruct + LoRA médico | Relatório clínico educacional |

Paths locais (`.env`):

```env
LOCAL_LLAMA_MODEL_PATH=./models/base/Meta-Llama-3-8B-Instruct
MEDICAL_ADAPTER_PATH=./models/llama3-8b-bnb-4bit-medical/adapter
```

Download dos pesos Hugging Face:

```powershell
python -m src.fine_tuning.download_local_llama
```

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv

ROOT = Path("..").resolve()
if not (ROOT / "requirements.txt").exists():
    ROOT = Path(".").resolve()

load_dotenv(ROOT / ".env")

paths = {
    "root": ROOT,
    "data_raw": ROOT / "data" / "raw",
    "data_processed": ROOT / "data" / "processed",
    "llama_base": ROOT / os.getenv("LOCAL_LLAMA_MODEL_PATH", "./models/base/Meta-Llama-3-8B-Instruct"),
    "medical_adapter": ROOT / os.getenv("MEDICAL_ADAPTER_PATH", "./models/llama3-8b-bnb-4bit-medical/adapter"),
}

for name, path in paths.items():
    exists = path.exists() if name != "root" else True
    print(f"{name:18} | exists={exists} | {path}")

## 6. Cenário do paciente fictício (fio condutor)

**Paciente:** J.S., 68 anos, pós-AVC, em reabilitação motora com monitoramento de UTI step-down.

| Turno | Modalidade | Evento simulado | Score esperado |
|---|---|---|---|
| Manhã | Vídeo — fisioterapia de marcha assistida | Assimetria de joelho / compensação de tronco | Risco motor ↑ |
| Tarde | Áudio — check-in de enfermagem | Fala arrastada, menção a “falta de ar” e fadiga | Risco de fala / sintomas ↑ |
| Noite | Vitais + prescrição | Queda de SpO₂ + taquicardia; dose de analgésico alterada fora do protocolo | Risco vital / prescrição ↑ |

**Fusão:** combinação ponderada dos três scores → risco global.  
**Motor de alertas:** LLM gera resumo em Markdown para a equipe (nível baixo / médio / alto).

Esse cenário único permite demonstrar no vídeo de entrega o fluxo completo: ingestão → modelos → fusão → alerta.

In [ ]:
# Esqueleto do paciente fictício (metadados do caso)
paciente = {
    "id": "JS-001",
    "idade": 68,
    "contexto": "pos-AVC / reabilitacao / UTI step-down",
    "modalidades": {
        "video": {
            "arquivo": "data/raw/video/js001_fisio_manha.mp4",
            "achado_esperado": "assimetria articular / compensacao de tronco",
        },
        "audio": {
            "arquivo": "data/raw/audio/js001_checkin_tarde.wav",
            "achado_esperado": "disartria leve + termos de fadiga/dispneia",
        },
        "vitals": {
            "arquivo": "data/raw/vitals/js001_noite.csv",
            "achado_esperado": "SpO2 baixa + taquicardia + desvio de prescricacao",
        },
    },
}

print(f"Caso {paciente['id']} — {paciente['contexto']}")
for mod, info in paciente["modalidades"].items():
    print(f"  [{mod}] {info['arquivo']}")
    print(f"         esperado: {info['achado_esperado']}")

## 7. Pipeline (visão operacional)

1. **Ingestão** dos dados brutos em `data/raw/{video,audio,vitals}`.
2. **Feature extraction** por modalidade → artefatos em `data/processed/`.
3. **Inferência** (`src/video`, `src/audio`, `src/vitals`) → score ∈ [0, 1] + achados.
4. **Fusão** (`src/fusion`) → risco global + breakdown.
5. **LLM + alerta** (`src/llm`, `src/alerts`) → resumo para a equipe.

**Tempo real:** no MVP, simulamos streaming por **janelas deslizantes** sobre as séries/vídeo/áudio (batch quase online), adequado para demo acadêmica sem infra de edge clínica.

## 8. Próximos experimentos (roadmap)

Notebooks futuros sugeridos (ainda não implementados nesta etapa):

| Notebook | Foco |
|---|---|
| `01_vitals_sinteticos.ipynb` | Gerar HR/SpO₂/PA + injetar anomalias; treinar Isolation Forest / PyOD |
| `02_audio_fala.ipynb` | Whisper + features de fadiga/disartria + termos críticos |
| `03_video_pose.ipynb` | MediaPipe/YOLOv8 em clipe próprio; desvio angular |
| `04_fusao_alertas.ipynb` | Combinar scores + Ollama/LoRA médico + notificação |

### Fora do escopo desta etapa

- Download completo PhysioNet/MIMIC
- Treino pesado / fine-tuning adicional
- Gravação definitiva dos clips de demo
- Implementação completa dos stubs em `src/*`
- Edição do vídeo YouTube/Vimeo (≤ 15 min)

In [ ]:
# Placeholder: checklist de experimentos (marcar conforme avanço)
experimentos = [
    {"id": "E1", "nome": "Vitais sintéticos + Isolation Forest", "feito": False},
    {"id": "E2", "nome": "Whisper + análise de fala", "feito": False},
    {"id": "E3", "nome": "Pose MediaPipe em vídeo próprio", "feito": False},
    {"id": "E4", "nome": "Fusão dos 3 scores", "feito": False},
    {"id": "E5", "nome": "Alerta LLM (Ollama / LoRA médico)", "feito": False},
]

for e in experimentos:
    mark = "[x]" if e["feito"] else "[ ]"
    print(f"{mark} {e['id']} — {e['nome']}")

## 9. Checklist de entrega (itens fáceis de esquecer)

Conforme o PDF da Fase 4 (atividade obrigatória, ~90% da nota das disciplinas da fase):

### Repositório Git
- [ ] Código-fonte completo da solução
- [ ] README com setup e arquitetura
- [ ] Dados de exemplo (ou script de geração) em `data/`
- [ ] Este relatório técnico evoluído com **resultados e exemplos de anomalias**

### Relatório técnico (conteúdo mínimo)
- [ ] Descrição do **fluxo multimodal**
- [ ] **Modelos** aplicados em cada tipo de dado
- [ ] **Resultados** obtidos e exemplos de anomalias detectadas
- [ ] Justificativa da substituição Azure → stack local
- [ ] Limitações (vídeo simulado, dados sintéticos, aviso ético)

### Vídeo demo (≤ 15 min · YouTube/Vimeo)
- [ ] Análise prática de **áudio e vídeo**
- [ ] Detecção e resposta a **anomalias**
- [ ] Explicação da “integração cloud” **via equivalência local** (ou Azure free, se usado)
- [ ] Fluxo final do **alerta à equipe médica**

### Detalhes técnicos frequentemente esquecidos
- [ ] Modalidade **texto** (prescrição / evolução clínica), além de áudio e vídeo
- [ ] Padrões de **movimentação** na internação (vídeo + features temporais)
- [ ] Definição operacional de **tempo real** (janelas)
- [ ] Aviso de que o LLM médico é educacional
- [ ] Evidências (prints/gráficos) dos três scores e do score fusionado

## 10. Resultados (a preencher)

_Esta seção será preenchida após os experimentos E1–E5._

| Modalidade | Métrica / evidência | Exemplo de anomalia |
|---|---|---|
| Vídeo | _pendente_ | _pendente_ |
| Áudio | _pendente_ | _pendente_ |
| Vitais | _pendente_ | _pendente_ |
| Fusão + alerta | _pendente_ | _pendente_ |

---

**Próximo passo sugerido:** implementar `01_vitals_sinteticos.ipynb` e a primeira versão de `src/vitals/anomaly_detection.py`.